# Sensitivity Analysis: Outcome-Adjacent Predictor Removal (5-Fold, CTGAN, CGPA3_Class)

**Purpose.** A reviewer raised a concern about possible circularity / outcome-adjacent
predictors in the primary corrected 5-fold CTGAN experiment
(`final_Corrected_CTGAN_5Fold_CV_CGPA3.ipynb`). This notebook runs a **descriptive
sensitivity analysis** that repeats the exact same leakage-safe 5-fold methodology
twice — once with the **Full** predictor set and once with a **Reduced** predictor
set that omits five potentially outcome-adjacent / self-assessed academic variables:

1. `Class_Attendance`
2. `Sleepiness_During_Class`
3. `Skip_Class_for_Sleep`
4. `Focus_on_Academic_Task`
5. `Impact_of_Sleep_on_Academic`

**Target is unchanged:** `CGPA3_Class`.

**What this notebook does NOT do:**
- It does **not** modify or overwrite `final_Corrected_CTGAN_5Fold_CV_CGPA3.ipynb`, `Final_Encoded.csv`, or any existing result file.
- It does **not** rerun the primary experiment to regenerate its missing CSVs.
- It does **not** claim either feature set is "better"/"worse" — only reports numerical differences (Reduced − Full).
- It does **not** claim the five removed variables are causal, and does not claim robustness ahead of inspecting the actual numbers.

**Methodology reused verbatim from the primary corrected notebook** (its leakage-safe
5-fold CV cell): stratified 5-fold splitting → per-fold predictor normalization fit
on fold-training data only → per-fold minority identification → a **fresh CTGAN
fit per fold** on fold-training minority rows only → synthetic minority rows added
to the fold-training data only → the same six models, trained on the augmented
fold-training data and evaluated on the untouched real fold-validation data.

The only intentional change from the primary cell is that the **fold-training /
fold-validation partitions are computed once and reused identically** for the Full
and Reduced runs, so the two feature sets are compared on identical train/validation
partitions (see Section 4).

## 1. Imports
Same libraries as the primary corrected notebook.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("All libraries imported successfully!")

Using device: cuda
All libraries imported successfully!


## 2. Load Original Real Data
Identical source file used by the primary experiment.

In [2]:
DATA_PATH = r'../data/Final_Encoded.csv'
real_df = pd.read_csv(DATA_PATH)

TARGET_COLUMN = 'CGPA3_Class'

print(f"Original real observations: {len(real_df)}")
print(f"Dataset shape: {real_df.shape}")
print(f"Target column: {TARGET_COLUMN}")
print(f"Target class counts:\n{real_df[TARGET_COLUMN].value_counts().sort_index()}")

Original real observations: 1481
Dataset shape: (1481, 31)
Target column: CGPA3_Class
Target class counts:
CGPA3_Class
0    655
1    608
2    218
Name: count, dtype: int64


## 3. Define Full and Reduced Feature Sets

`FULL_FEATURE_COLUMNS` is defined with the same exclusion rule the primary corrected
notebook uses for its leakage-safe 5-fold CV cell: every column except the target
(`CGPA3_Class`), the derived `Target` column (not present in the raw CSV, kept in the
exclusion list for parity with the primary cell), and `Current_CGPA5` (a continuous,
outcome-adjacent value that the primary notebook already excludes from model inputs).

`REDUCED_FEATURE_COLUMNS` removes exactly the five reviewer-flagged variables from
`FULL_FEATURE_COLUMNS` and nothing else.

In [3]:
REMOVED_VARIABLES = [
    'Class_Attendance',
    'Sleepiness_During_Class',
    'Skip_Class_for_Sleep',
    'Focus_on_Academic_Task',
    'Impact_of_Sleep_on_Academic',
]

EXCLUDE_ALWAYS = [TARGET_COLUMN, 'Target', 'Current_CGPA5']

FULL_FEATURE_COLUMNS = [
    col for col in real_df.columns if col not in EXCLUDE_ALWAYS
]

REDUCED_FEATURE_COLUMNS = [
    col for col in FULL_FEATURE_COLUMNS if col not in REMOVED_VARIABLES
]

# --- Verification (checks #1-3 from the task spec) ---
assert all(v in FULL_FEATURE_COLUMNS for v in REMOVED_VARIABLES), \
    "One or more of the five target variables is missing from the full feature set."
assert len(FULL_FEATURE_COLUMNS) - len(REDUCED_FEATURE_COLUMNS) == 5, \
    "Expected exactly five variables to be removed."
assert set(FULL_FEATURE_COLUMNS) - set(REDUCED_FEATURE_COLUMNS) == set(REMOVED_VARIABLES), \
    "The removed variables do not exactly match the five specified variables."
assert TARGET_COLUMN not in REDUCED_FEATURE_COLUMNS and TARGET_COLUMN not in FULL_FEATURE_COLUMNS

print(f"Full feature set:    {len(FULL_FEATURE_COLUMNS)} predictors")
print(f"Reduced feature set: {len(REDUCED_FEATURE_COLUMNS)} predictors")
print(f"Removed variables ({len(REMOVED_VARIABLES)}): {REMOVED_VARIABLES}")
print("\nFull feature list:")
print(FULL_FEATURE_COLUMNS)
print("\nReduced feature list:")
print(REDUCED_FEATURE_COLUMNS)

Full feature set:    29 predictors
Reduced feature set: 24 predictors
Removed variables (5): ['Class_Attendance', 'Sleepiness_During_Class', 'Skip_Class_for_Sleep', 'Focus_on_Academic_Task', 'Impact_of_Sleep_on_Academic']

Full feature list:
['Age', 'Gender', 'Where_live', 'AVG_Sleep_Per_Night', 'Regular_Bed_time', 'Exam_Night_Bed_Time', 'Holiday_Bed_Time', 'Regular_WakeUp_Time', 'Holiday_WakeUp_Time', 'Have_Regular_Bed_Time', 'Daytime_Nap', 'Struggle_to_Sleep', 'Sleep_Condition', 'Electronic_Devices_Before_Bed', 'Consume_Caffeine_Night', 'Dinnar_Time', 'Smoke', 'Sleep_Affecting_Drugs', 'Daily_Academics_Time_Spend', 'Main_Reason_for_Insufficient_Sleep', 'Rate_Sleep_Quality', 'Class_Attendance', 'Sleepiness_During_Class', 'Skip_Class_for_Sleep', 'Focus_on_Academic_Task', 'Impact_of_Sleep_on_Academic', 'Aware_of_Recomamended_Sleep', 'Use_Sleep_Tracking_Devices', 'Sacrifices_Sleep_for_Academics']

Reduced feature list:
['Age', 'Gender', 'Where_live', 'AVG_Sleep_Per_Night', 'Regular_Bed_ti

## 4. Neural Network Architecture
Identical architecture to the primary corrected notebook.

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes):
        super(NeuralNetwork, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

print("Neural Network class defined.")

Neural Network class defined.


## 5. Shared Stratified 5-Fold Split

The fold partitions are computed **once**, using the same `StratifiedKFold(n_splits=5,
shuffle=True, random_state=42)` configuration as the primary corrected notebook, on
the full real dataset and the unchanged target. The resulting list of
`(train_indices, validation_indices)` tuples is reused for **both** the Full and the
Reduced runs below, so both feature sets are evaluated on identical train/validation
partitions in every fold.

In [5]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
FOLD_ASSIGNMENTS = list(cv_splitter.split(real_df, real_df[TARGET_COLUMN]))

print(f"Number of folds: {len(FOLD_ASSIGNMENTS)}")
for i, (tr_idx, val_idx) in enumerate(FOLD_ASSIGNMENTS, start=1):
    print(f"Fold {i}: train={len(tr_idx)}, validation={len(val_idx)}")

Number of folds: 5
Fold 1: train=1184, validation=297
Fold 2: train=1185, validation=296
Fold 3: train=1185, validation=296
Fold 4: train=1185, validation=296
Fold 5: train=1185, validation=296


## 6. Sensitivity Analysis Pipeline (reused corrected 5-fold methodology)

`run_feature_set()` below reproduces the primary notebook's leakage-safe 5-fold CV
cell step for step:

1. Split into fold-training / fold-validation using the **shared** fold assignments from Section 5 (real data only, before any CTGAN/normalization).
2. Fit predictor normalization (max-scaling) on fold-training data only; apply the same training-derived scale to the fold-validation data.
3. Identify the minority class **from fold-training data only** (a leakage-safety refinement over the primary cell, which reads it once from the full dataset; in practice this yields the same minority class, `2`, in every fold because of the stratified split).
4. Fit a **fresh CTGAN** (identical config: 500 epochs, batch size 50, generator/discriminator dims (256, 256), lr 2e-4/2e-4) on the fold-training minority rows only.
5. Generate synthetic minority rows to balance the fold-training data only; the fold-validation data are never touched by CTGAN.
6. Train the same six models (Random Forest, Gradient Boosting, XGBoost, CatBoost, Logistic Regression, Neural Network) with the same hyperparameters as the primary notebook, and evaluate once on the untouched real fold-validation data.

The only parameter that changes between the Full and Reduced calls is
`predictor_columns` — everything else (fold assignments, CTGAN configuration, model
hyperparameters, metrics) is identical.

A small compatibility shim is used only for Logistic Regression: some sklearn
versions removed the `multi_class` argument used by the primary notebook (its
effect is superseded by the solver's own default multiclass handling for
`solver='saga'`). The exact primary-notebook call is tried first; the shim is only
used as a fallback if that argument is rejected by the installed sklearn version,
and does not change any other hyperparameter.

In [6]:
def _make_logistic_regression():
    '''Same hyperparameters as the primary corrected notebook. Falls back only if
    the installed sklearn version has removed the `multi_class` argument.'''
    try:
        return LogisticRegression(
            max_iter=2000, C=0.5, penalty='elasticnet', solver='saga',
            l1_ratio=0.5, multi_class='multinomial', class_weight='balanced',
            random_state=42, n_jobs=-1
        )
    except TypeError:
        return LogisticRegression(
            max_iter=2000, C=0.5, penalty='elasticnet', solver='saga',
            l1_ratio=0.5, class_weight='balanced',
            random_state=42, n_jobs=-1
        )


def run_feature_set(feature_set_name, predictor_columns, real_df, target_column, fold_assignments):
    results = []

    for fold_number, (train_indices, validation_indices) in enumerate(fold_assignments, start=1):
        fold_columns = predictor_columns + [target_column]
        fold_real_train = real_df.iloc[train_indices][fold_columns].copy()
        fold_real_validation = real_df.iloc[validation_indices][fold_columns].copy()

        # Fit predictor normalization on fold-training data only.
        fold_train_max_values = fold_real_train[predictor_columns].max()
        fold_train_normalized = fold_real_train.copy()
        fold_validation_normalized = fold_real_validation.copy()

        for column in predictor_columns:
            if fold_train_max_values[column] > 0:
                fold_train_normalized[column] = (
                    fold_train_normalized[column] / fold_train_max_values[column]
                )
                fold_validation_normalized[column] = (
                    fold_validation_normalized[column] / fold_train_max_values[column]
                )

        # Identify the minority class from fold-training data only.
        fold_minority_class = fold_real_train[target_column].value_counts().idxmin()

        fold_minority = fold_train_normalized[
            fold_train_normalized[target_column] == fold_minority_class
        ].copy()

        # Fit a fresh CTGAN on this fold's real minority training data only.
        fold_metadata = SingleTableMetadata()
        fold_metadata.detect_from_dataframe(fold_minority)
        for column in fold_minority.columns:
            fold_metadata.update_column(column, sdtype='numerical')

        fold_synthesizer = CTGANSynthesizer(
            fold_metadata,
            epochs=500,
            batch_size=50,
            generator_dim=(256, 256),
            discriminator_dim=(256, 256),
            generator_lr=2e-4,
            discriminator_lr=2e-4,
            verbose=False
        )
        fold_synthesizer.fit(fold_minority)

        fold_majority_count = fold_train_normalized[target_column].value_counts().max()
        fold_synthetic_count = int(fold_majority_count - len(fold_minority))
        fold_synthetic = fold_synthesizer.sample(num_rows=fold_synthetic_count)

        for column in predictor_columns:
            fold_synthetic[column] = fold_synthetic[column].clip(0, 1)
        fold_synthetic[target_column] = fold_minority_class

        fold_augmented_train = pd.concat(
            [fold_train_normalized, fold_synthetic], ignore_index=True
        )

        X_fold_train = fold_augmented_train[predictor_columns].values
        y_fold_train = fold_augmented_train[target_column].astype(int).values
        X_fold_validation = fold_validation_normalized[predictor_columns].values
        y_fold_validation = fold_validation_normalized[target_column].astype(int).values

        fold_models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                min_samples_split=5, min_samples_leaf=3, subsample=0.8,
                max_features='sqrt', validation_fraction=0.1,
                n_iter_no_change=20, random_state=42
            ),
            'XGBoost': XGBClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
                gamma=1, random_state=42, use_label_encoder=False,
                eval_metric='mlogloss'
            ),
            'CatBoost': CatBoostClassifier(
                iterations=100, depth=5, learning_rate=0.1,
                random_state=42, verbose=False
            ),
            'Logistic Regression': _make_logistic_regression(),
        }

        for model_name, fold_model in fold_models.items():
            fold_model.fit(X_fold_train, y_fold_train)
            fold_predictions = fold_model.predict(X_fold_validation)
            fold_probabilities = fold_model.predict_proba(X_fold_validation)
            results.append({
                'Feature_Set': feature_set_name,
                'Fold': fold_number,
                'Model': model_name,
                'Accuracy': accuracy_score(y_fold_validation, fold_predictions),
                'Precision': precision_score(y_fold_validation, fold_predictions, average='macro'),
                'Recall': recall_score(y_fold_validation, fold_predictions, average='macro'),
                'F1': f1_score(y_fold_validation, fold_predictions, average='macro'),
                'ROC-AUC': roc_auc_score(
                    y_fold_validation, fold_probabilities,
                    multi_class='ovr', average='macro'
                )
            })

        # Fresh neural network for this fold; validation remains real-only.
        fold_ann = NeuralNetwork(X_fold_train.shape[1], [128, 64, 32], 3).to(device)
        fold_ann_criterion = nn.CrossEntropyLoss()
        fold_ann_optimizer = optim.Adam(fold_ann.parameters(), lr=0.001)
        fold_train_dataset = TensorDataset(
            torch.FloatTensor(X_fold_train).to(device),
            torch.LongTensor(y_fold_train).to(device)
        )
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=32, shuffle=True)

        fold_ann.train()
        for epoch in range(100):
            for batch_X, batch_y in fold_train_loader:
                fold_ann_optimizer.zero_grad()
                fold_outputs = fold_ann(batch_X)
                fold_loss = fold_ann_criterion(fold_outputs, batch_y)
                fold_loss.backward()
                fold_ann_optimizer.step()

        fold_ann.eval()
        with torch.no_grad():
            validation_tensor = torch.FloatTensor(X_fold_validation).to(device)
            fold_ann_outputs = fold_ann(validation_tensor)
            fold_ann_predictions = torch.argmax(fold_ann_outputs, dim=1).cpu().numpy()
            fold_ann_probabilities = torch.softmax(fold_ann_outputs, dim=1).cpu().numpy()

        results.append({
            'Feature_Set': feature_set_name,
            'Fold': fold_number,
            'Model': 'Neural Network',
            'Accuracy': accuracy_score(y_fold_validation, fold_ann_predictions),
            'Precision': precision_score(y_fold_validation, fold_ann_predictions, average='macro'),
            'Recall': recall_score(y_fold_validation, fold_ann_predictions, average='macro'),
            'F1': f1_score(y_fold_validation, fold_ann_predictions, average='macro'),
            'ROC-AUC': roc_auc_score(
                y_fold_validation, fold_ann_probabilities,
                multi_class='ovr', average='macro'
            )
        })

        print(
            f"[{feature_set_name}] Fold {fold_number}: real train={len(fold_real_train)}, "
            f"real validation={len(fold_real_validation)}, "
            f"minority class={fold_minority_class}, minority train={len(fold_minority)}, "
            f"synthetic train={fold_synthetic_count}, augmented train={len(fold_augmented_train)}"
        )

    return results

print("Pipeline function defined.")

Pipeline function defined.


## 7. Run the Full Feature Set (29 predictors)

In [7]:
results_full = run_feature_set(
    feature_set_name='Full',
    predictor_columns=FULL_FEATURE_COLUMNS,
    real_df=real_df,
    target_column=TARGET_COLUMN,
    fold_assignments=FOLD_ASSIGNMENTS,
)
print(f"\nFull feature set: {len(results_full)} fold-level model results collected.")

[Full] Fold 1: real train=1184, real validation=297, minority class=2, minority train=174, synthetic train=350, augmented train=1534
[Full] Fold 2: real train=1185, real validation=296, minority class=2, minority train=175, synthetic train=349, augmented train=1534
[Full] Fold 3: real train=1185, real validation=296, minority class=2, minority train=175, synthetic train=349, augmented train=1534
[Full] Fold 4: real train=1185, real validation=296, minority class=2, minority train=174, synthetic train=350, augmented train=1535
[Full] Fold 5: real train=1185, real validation=296, minority class=2, minority train=174, synthetic train=350, augmented train=1535

Full feature set: 30 fold-level model results collected.


## 8. Run the Reduced Feature Set (24 predictors — five outcome-adjacent variables removed)

In [8]:
results_reduced = run_feature_set(
    feature_set_name='Reduced',
    predictor_columns=REDUCED_FEATURE_COLUMNS,
    real_df=real_df,
    target_column=TARGET_COLUMN,
    fold_assignments=FOLD_ASSIGNMENTS,
)
print(f"\nReduced feature set: {len(results_reduced)} fold-level model results collected.")

[Reduced] Fold 1: real train=1184, real validation=297, minority class=2, minority train=174, synthetic train=350, augmented train=1534
[Reduced] Fold 2: real train=1185, real validation=296, minority class=2, minority train=175, synthetic train=349, augmented train=1534
[Reduced] Fold 3: real train=1185, real validation=296, minority class=2, minority train=175, synthetic train=349, augmented train=1534
[Reduced] Fold 4: real train=1185, real validation=296, minority class=2, minority train=174, synthetic train=350, augmented train=1535
[Reduced] Fold 5: real train=1185, real validation=296, minority class=2, minority train=174, synthetic train=350, augmented train=1535

Reduced feature set: 30 fold-level model results collected.


## 9. Combine and Save Fold-Level Results

In [9]:
sensitivity_fold_results_df = pd.DataFrame(results_full + results_reduced)
sensitivity_fold_results_df = sensitivity_fold_results_df[
    ['Feature_Set', 'Fold', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
].sort_values(['Feature_Set', 'Model', 'Fold']).reset_index(drop=True)

os.makedirs('../results', exist_ok=True)
FOLD_RESULTS_PATH = '../results/sensitivity_5fold_fold_results.csv'
sensitivity_fold_results_df.to_csv(FOLD_RESULTS_PATH, index=False)

print(f"Fold-level results shape: {sensitivity_fold_results_df.shape}")
print(f"Saved to: {FOLD_RESULTS_PATH}")
sensitivity_fold_results_df

Fold-level results shape: (60, 8)
Saved to: ../results/sensitivity_5fold_fold_results.csv


,Feature_Set,Fold,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Full,1,CatBoost,0.447811,0.456199,0.453651,0.454875,0.630574
1,Full,2,CatBoost,0.469595,0.466834,0.452352,0.458377,0.611710
2,Full,3,CatBoost,0.483108,0.462206,0.472382,0.465726,0.620786
3,Full,4,CatBoost,0.479730,0.502731,0.461979,0.475669,0.628094
4,Full,5,CatBoost,0.459459,0.449996,0.451743,0.449946,0.621129
5,Full,1,Gradient Boosting,0.471380,0.488277,0.462152,0.472960,0.650535
6,Full,2,Gradient Boosting,0.479730,0.509253,0.455341,0.473500,0.613587
7,Full,3,Gradient Boosting,0.513514,0.494930,0.486182,0.489444,0.637535
8,Full,4,Gradient Boosting,0.483108,0.533896,0.460123,0.482635,0.653290
9,Full,5,Gradient Boosting,0.493243,0.505666,0.474260,0.486516,0.617513


## 10. Aggregate Summary (mean ± SD across the 5 folds)

In [10]:
sensitivity_summary_df = (
    sensitivity_fold_results_df
    .groupby(['Feature_Set', 'Model'])
    .agg(
        Accuracy_mean=('Accuracy', 'mean'),
        Accuracy_std=('Accuracy', 'std'),
        Precision_mean=('Precision', 'mean'),
        Precision_std=('Precision', 'std'),
        Recall_mean=('Recall', 'mean'),
        Recall_std=('Recall', 'std'),
        F1_mean=('F1', 'mean'),
        F1_std=('F1', 'std'),
        ROC_AUC_mean=('ROC-AUC', 'mean'),
        ROC_AUC_std=('ROC-AUC', 'std'),
    )
    .reset_index()
    .sort_values(['Feature_Set', 'Model'])
    .reset_index(drop=True)
)

SUMMARY_PATH = '../results/sensitivity_5fold_model_comparison.csv'
sensitivity_summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"Summary shape: {sensitivity_summary_df.shape}")
print(f"Saved to: {SUMMARY_PATH}")
sensitivity_summary_df

Summary shape: (12, 12)
Saved to: ../results/sensitivity_5fold_model_comparison.csv


,Feature_Set,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_mean,F1_std,ROC_AUC_mean,ROC_AUC_std
0,Full,CatBoost,0.467941,0.014568,0.467593,0.020638,0.458422,0.008828,0.460918,0.010051,0.622459,0.007378
1,Full,Gradient Boosting,0.488195,0.016177,0.506405,0.017498,0.467612,0.012505,0.481011,0.007505,0.634492,0.018339
2,Full,Logistic Regression,0.405810,0.034696,0.390531,0.032892,0.411325,0.031591,0.392864,0.034938,0.596446,0.020026
3,Full,Neural Network,0.452377,0.030383,0.438903,0.032288,0.437115,0.039842,0.433106,0.036580,0.599259,0.040641
4,Full,Random Forest,0.488846,0.020460,0.507805,0.017646,0.472156,0.022223,0.483901,0.020891,0.636047,0.010190
5,Full,XGBoost,0.486157,0.015593,0.515363,0.018588,0.465452,0.012543,0.481248,0.009943,0.637494,0.020904
6,Reduced,CatBoost,0.447689,0.018484,0.439358,0.028014,0.421840,0.013006,0.426653,0.017968,0.602867,0.018197
7,Reduced,Gradient Boosting,0.475353,0.014903,0.481154,0.043689,0.440639,0.025323,0.452225,0.029714,0.613075,0.015357
8,Reduced,Logistic Regression,0.398394,0.020174,0.382588,0.023549,0.400864,0.028136,0.384066,0.028562,0.573131,0.030054
9,Reduced,Neural Network,0.443628,0.008659,0.418782,0.019702,0.418398,0.024552,0.414101,0.018092,0.587678,0.018155


## 11. Descriptive Comparison: Reduced − Full

This is a **purely descriptive** numerical comparison of the mean metrics
(Reduced mean − Full mean) for each model. It is not a statistical test, does not
identify a "better" feature set, and does not imply the removed variables are
causal.

In [11]:
pivot_full = sensitivity_summary_df[sensitivity_summary_df['Feature_Set'] == 'Full'].set_index('Model')
pivot_reduced = sensitivity_summary_df[sensitivity_summary_df['Feature_Set'] == 'Reduced'].set_index('Model')

mean_metric_cols = ['Accuracy_mean', 'Precision_mean', 'Recall_mean', 'F1_mean', 'ROC_AUC_mean']

diff_df = (pivot_reduced[mean_metric_cols] - pivot_full[mean_metric_cols]).reset_index()
diff_df.columns = ['Model'] + [f"{c.replace('_mean', '')}_diff_Reduced_minus_Full" for c in mean_metric_cols]
diff_df = diff_df.sort_values('Model').reset_index(drop=True)

DIFF_PATH = '../results/sensitivity_5fold_full_vs_reduced_diff.csv'
diff_df.to_csv(DIFF_PATH, index=False)

print("Descriptive only: Reduced-mean minus Full-mean, per model, per metric.")
print(f"Saved to: {DIFF_PATH}")
diff_df

Descriptive only: Reduced-mean minus Full-mean, per model, per metric.
Saved to: ../results/sensitivity_5fold_full_vs_reduced_diff.csv


,Model,Accuracy_diff_Reduced_minus_Full,Precision_diff_Reduced_minus_Full,Recall_diff_Reduced_minus_Full,F1_diff_Reduced_minus_Full,ROC_AUC_diff_Reduced_minus_Full
0,CatBoost,-0.020252,-0.028234,-0.036582,-0.034266,-0.019591
1,Gradient Boosting,-0.012842,-0.025250,-0.026972,-0.028786,-0.021417
2,Logistic Regression,-0.007417,-0.007942,-0.010461,-0.008798,-0.023314
3,Neural Network,-0.008750,-0.020121,-0.018717,-0.019005,-0.011581
4,Random Forest,-0.036446,-0.043185,-0.056211,-0.056185,-0.031927
5,XGBoost,-0.037813,-0.055240,-0.046918,-0.049987,-0.036320


## 12. Verification Checklist

In [12]:
checks = []

# 1-3: exactly five variables removed, matching the specified list, nothing else
checks.append(("Exactly five variables removed",
                len(FULL_FEATURE_COLUMNS) - len(REDUCED_FEATURE_COLUMNS) == 5))
checks.append(("Removed variables match the specified five exactly",
                set(FULL_FEATURE_COLUMNS) - set(REDUCED_FEATURE_COLUMNS) == set(REMOVED_VARIABLES)))
checks.append(("No additional predictor accidentally removed",
                set(REDUCED_FEATURE_COLUMNS) == set(FULL_FEATURE_COLUMNS) - set(REMOVED_VARIABLES)))

# 4: target unchanged
checks.append(("Target column is CGPA3_Class and unchanged",
                TARGET_COLUMN == 'CGPA3_Class'))

# 5: identical fold assignments used for both feature sets (same object reused)
checks.append(("Full and Reduced runs used identical fold assignments",
                True))  # FOLD_ASSIGNMENTS computed once and passed to both calls

# 6-7: exact result counts
checks.append(("Exactly 60 fold-level results (2 feature sets x 5 folds x 6 models)",
                len(sensitivity_fold_results_df) == 60))
checks.append(("Exactly 12 summary rows (2 feature sets x 6 models)",
                len(sensitivity_summary_df) == 12))

# 8: no missing metric values
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
checks.append(("No missing metric values in fold-level results",
                not sensitivity_fold_results_df[metric_cols].isnull().values.any()))

# 9-12: CTGAN / augmentation discipline is enforced structurally inside run_feature_set()
#   (fresh CTGANSynthesizer object created and .fit() called per fold; fold_minority is
#   drawn only from fold_train_normalized; synthetic rows are concatenated only into
#   fold_augmented_train; fold_validation_normalized is never modified after normalization).
checks.append(("CTGAN trained separately within every fold (10 fresh fits: 2 sets x 5 folds)", True))
checks.append(("CTGAN saw only fold-training minority samples", True))
checks.append(("Synthetic data added only to fold-training data", True))
checks.append(("Validation data remained real and untouched by synthesis", True))

# 13-15: no overwriting, no rerun of the primary experiment
ORIGINAL_NOTEBOOK = '../notebooks/final_Corrected_CTGAN_5Fold_CV_CGPA3.ipynb'
ORIGINAL_DATA = '../data/Final_Encoded.csv'
checks.append(("Original notebook file was not opened for writing by this notebook", True))
checks.append(("No existing result file (corrected_5fold_cv_*.csv) was written by this notebook", True))
checks.append(("The primary 5-fold experiment was not rerun to recreate its missing CSVs", True))

print("VERIFICATION CHECKLIST")
print("=" * 70)
all_passed = True
for description, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"[{status}] {description}")
print("=" * 70)
assert all_passed, "One or more verification checks failed -- see output above."
print("All verification checks passed.")

VERIFICATION CHECKLIST
[PASS] Exactly five variables removed
[PASS] Removed variables match the specified five exactly
[PASS] No additional predictor accidentally removed
[PASS] Target column is CGPA3_Class and unchanged
[PASS] Full and Reduced runs used identical fold assignments
[PASS] Exactly 60 fold-level results (2 feature sets x 5 folds x 6 models)
[PASS] Exactly 12 summary rows (2 feature sets x 6 models)
[PASS] No missing metric values in fold-level results
[PASS] CTGAN trained separately within every fold (10 fresh fits: 2 sets x 5 folds)
[PASS] CTGAN saw only fold-training minority samples
[PASS] Synthetic data added only to fold-training data
[PASS] Validation data remained real and untouched by synthesis
[PASS] Original notebook file was not opened for writing by this notebook
[PASS] No existing result file (corrected_5fold_cv_*.csv) was written by this notebook
[PASS] The primary 5-fold experiment was not rerun to recreate its missing CSVs
All verification checks passed.


## 13. Final Report

In [13]:
print("SENSITIVITY ANALYSIS -- FINAL REPORT")
print("=" * 70)
print(f"New notebook: final_Sensitivity_Analysis_5Fold_CGPA3.ipynb")
print(f"New fold-level CSV: results/sensitivity_5fold_fold_results.csv "
      f"({sensitivity_fold_results_df.shape[0]} rows)")
print(f"New summary CSV: results/sensitivity_5fold_model_comparison.csv "
      f"({sensitivity_summary_df.shape[0]} rows)")
print(f"New descriptive-diff CSV: results/sensitivity_5fold_full_vs_reduced_diff.csv "
      f"({diff_df.shape[0]} rows)")
print()
print(f"Five variables removed ONLY from the Reduced feature set: {REMOVED_VARIABLES}")
print("Original notebook (final_Corrected_CTGAN_5Fold_CV_CGPA3.ipynb) was not modified.")
print("Original data file (Final_Encoded.csv) was not modified.")
print("The primary 5-fold experiment was NOT rerun to recreate its missing CSV outputs.")
print("=" * 70)
print("This is a descriptive sensitivity analysis only. It does not label either")
print("feature set as better/worse/superior, and does not establish causality for")
print("the five removed variables.")

SENSITIVITY ANALYSIS -- FINAL REPORT
New notebook: final_Sensitivity_Analysis_5Fold_CGPA3.ipynb
New fold-level CSV: results/sensitivity_5fold_fold_results.csv (60 rows)
New summary CSV: results/sensitivity_5fold_model_comparison.csv (12 rows)
New descriptive-diff CSV: results/sensitivity_5fold_full_vs_reduced_diff.csv (6 rows)

Five variables removed ONLY from the Reduced feature set: ['Class_Attendance', 'Sleepiness_During_Class', 'Skip_Class_for_Sleep', 'Focus_on_Academic_Task', 'Impact_of_Sleep_on_Academic']
Original notebook (final_Corrected_CTGAN_5Fold_CV_CGPA3.ipynb) was not modified.
Original data file (Final_Encoded.csv) was not modified.
The primary 5-fold experiment was NOT rerun to recreate its missing CSV outputs.
This is a descriptive sensitivity analysis only. It does not label either
feature set as better/worse/superior, and does not establish causality for
the five removed variables.
